# Smart Frame Selection for Logo Annotation
## Bradford Bulls — AI Sponsorship Exposure Valuation

### Problem
A 1.5h match video @ 30FPS = 162,000 frames. Naive sampling still gives 6,000+ frames — too many to annotate, and most have blurry/tiny logos.

### Solution: Quality-First Selection
Instead of "sample → filter", we **score every candidate frame** and pick the **Top N best frames** where logos are most clearly visible.

**Scoring formula per frame:**
| Factor | What it measures | Why it matters |
|--------|-----------------|----------------|
| Player Size | Largest player bbox as % of frame | Bigger player = bigger logo = easier to see & annotate |
| Jersey Sharpness | Laplacian variance on cropped jersey region | Blurry jersey = unreadable logo |
| Frontality | Aspect ratio of player bbox (tall+narrow = front) | Front view shows chest/collar logos |
| Frame Position | Player near center of frame | Broadcast tends to focus camera on key action |

**Pipeline:**
1. Download video (yt-dlp)
2. Temporal sample at 2 FPS (~10K frames)
3. YOLOv8 person detection on ALL sampled frames (GPU fast)
4. Score each frame → rank by Logo Visibility Score
5. Cluster top frames to remove near-duplicates
6. Export Top 300-500 best frames for annotation

**Target: ~300 high-quality, diverse frames with clearly visible logos**

> Designed to run on **Google Colab** with GPU runtime.

## 1. Setup (Colab)

In [ ]:
# ============================================================
# 1A. Install dependencies
# ============================================================
!pip install -q ultralytics yt-dlp imagehash scikit-image roboflow pillow opencv-python-headless

# Verify GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

In [ ]:
# ============================================================
# 1B. Mount Google Drive — ALL data lives here
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, csv, json
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import imagehash
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO

# ============================================================
# ALL PATHS ON GOOGLE DRIVE
# ============================================================
# Change this root to match your Drive folder structure
DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")

VIDEOS_DIR       = DRIVE_ROOT / "videos"
ALL_FRAMES_DIR   = DRIVE_ROOT / "all_frames"
BEST_FRAMES_DIR  = DRIVE_ROOT / "best_frames"
METADATA_DIR     = DRIVE_ROOT / "metadata"
AUTOLABEL_DIR    = DRIVE_ROOT / "auto_annotated"
DATASET_DIR      = DRIVE_ROOT / "dataset"

for d in [VIDEOS_DIR, ALL_FRAMES_DIR, BEST_FRAMES_DIR, METADATA_DIR, AUTOLABEL_DIR, DATASET_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"All data stored on Google Drive:")
print(f"  Root:         {DRIVE_ROOT}")
print(f"  Videos:       {VIDEOS_DIR}")
print(f"  Best frames:  {BEST_FRAMES_DIR}")
print(f"  Metadata:     {METADATA_DIR}")
print(f"  Auto-labels:  {AUTOLABEL_DIR}")

## 2. Download Video

In [ ]:
# ============================================================
# VIDEO SOURCE — change this URL for different matches
# ============================================================
VIDEO_URL = "https://www.youtube.com/watch?v=Yly5ELzUmbw"

# Download directly to Google Drive (persists across sessions)
!yt-dlp -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best" \
        --merge-output-format mp4 \
        -o "{VIDEOS_DIR}/%(title)s.%(ext)s" \
        --no-playlist \
        "{VIDEO_URL}"

# Find downloaded file
video_files = sorted(VIDEOS_DIR.glob("*.mp4"), key=lambda f: f.stat().st_mtime, reverse=True)
if not video_files:
    video_files = sorted(VIDEOS_DIR.glob("*.*"), key=lambda f: f.stat().st_mtime, reverse=True)
VIDEO_PATH = video_files[0]

# Get metadata
cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = TOTAL_FRAMES / FPS
cap.release()

print(f"\nVideo: {VIDEO_PATH.name}")
print(f"Saved to: {VIDEO_PATH}")
print(f"Duration: {DURATION/60:.1f} min | Resolution: {WIDTH}x{HEIGHT} | FPS: {FPS:.0f}")
print(f"Total frames: {TOTAL_FRAMES:,}")
print(f"\nNote: Video is on Google Drive — no need to re-download next session!")

## 3. Phase 1 — Temporal Sampling + Person Detection (GPU Batch)

Extract frames at 2 FPS and run YOLOv8 person detection on **every** frame in GPU batches.
This is fast on Colab GPU: ~1000 frames/min with batch inference.

We **don't save frames to disk yet** — we only compute scores. Frames are saved later (only the best ones).

In [ ]:
# ============================================================
# CONFIG
# ============================================================
TARGET_FPS = 2          # Temporal sampling rate
TARGET_FRAMES = 300     # How many best frames to select for annotation
YOLO_BATCH_SIZE = 32    # GPU batch size for detection (adjust if OOM)
YOLO_MODEL = "yolov8m.pt"  # medium model — good balance of speed/accuracy

# ============================================================
# STEP 1: Extract frames at TARGET_FPS (in memory, not disk)
# ============================================================
frame_interval = max(1, int(FPS / TARGET_FPS))
estimated_samples = TOTAL_FRAMES // frame_interval
print(f"Sampling every {frame_interval} frames ({TARGET_FPS} FPS)")
print(f"Estimated candidate frames: {estimated_samples:,}")

# Load YOLOv8 model
model = YOLO(YOLO_MODEL)
print(f"YOLOv8 model loaded: {YOLO_MODEL}")

In [ ]:
# ============================================================
# STEP 2: Score every sampled frame
# ============================================================
# Process in batches for GPU efficiency.
# For each frame, compute a "Logo Visibility Score" (LVS):
#
#   LVS = player_size_score × 0.40     (biggest factor: large player = visible logo)
#       + jersey_sharpness   × 0.25     (sharp jersey region = readable logo)
#       + frontality_score   × 0.20     (front-facing = chest logos visible)
#       + center_score       × 0.15     (player near center = broadcast focus)
#

def compute_jersey_sharpness(frame, bbox):
    """Compute sharpness of the upper body region (where most logos are)."""
    x1, y1, x2, y2 = [int(v) for v in bbox]
    h_box = y2 - y1
    # Jersey = upper 60% of person bbox (skip head top 15%, skip legs bottom 25%)
    jersey_y1 = y1 + int(h_box * 0.15)
    jersey_y2 = y1 + int(h_box * 0.60)
    jersey_y1 = max(0, jersey_y1)
    jersey_y2 = min(frame.shape[0], jersey_y2)
    x1 = max(0, x1)
    x2 = min(frame.shape[1], x2)

    if jersey_y2 <= jersey_y1 or x2 <= x1:
        return 0.0

    jersey = frame[jersey_y1:jersey_y2, x1:x2]
    gray = cv2.cvtColor(jersey, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def compute_frontality(bbox):
    """Estimate if player is front-facing based on bbox aspect ratio.
    Front-facing: tall and narrower. Side-facing: wider and shorter.
    """
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    if w == 0:
        return 0.0
    aspect = h / w
    # Front-facing players typically have aspect ratio 2.0-3.5
    # Side-running players are wider, aspect ratio 1.0-1.8
    if aspect >= 2.5:
        return 1.0
    elif aspect >= 1.8:
        return 0.7
    elif aspect >= 1.3:
        return 0.4
    else:
        return 0.2


def compute_center_score(bbox, frame_w, frame_h):
    """Score based on how close the player is to frame center."""
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    # Normalized distance from center (0 = center, 1 = corner)
    dx = abs(cx - frame_w / 2) / (frame_w / 2)
    dy = abs(cy - frame_h / 2) / (frame_h / 2)
    dist = (dx**2 + dy**2) ** 0.5 / (2**0.5)
    return 1.0 - dist


def score_frame(frame, detections, frame_w, frame_h):
    """
    Compute Logo Visibility Score for a frame.
    Returns (score, best_player_info) for the most promising player.
    """
    if not detections or len(detections) == 0:
        return 0.0, None

    frame_area = frame_w * frame_h
    best_score = 0.0
    best_info = None

    for det in detections:
        bbox = det[:4]
        conf = det[4]
        x1, y1, x2, y2 = bbox

        # Player size as fraction of frame
        player_area = (x2 - x1) * (y2 - y1)
        size_ratio = player_area / frame_area

        # Skip tiny players (< 1% of frame)
        if size_ratio < 0.01:
            continue

        # Normalize size score (0-1), cap at 25% of frame
        size_score = min(size_ratio / 0.25, 1.0)

        # Jersey sharpness (normalize to 0-1, typical range 0-500)
        sharpness = compute_jersey_sharpness(frame, bbox)
        sharpness_score = min(sharpness / 300.0, 1.0)

        # Frontality
        front_score = compute_frontality(bbox)

        # Center position
        center = compute_center_score(bbox, frame_w, frame_h)

        # Weighted composite score
        lvs = (
            size_score      * 0.40 +
            sharpness_score * 0.25 +
            front_score     * 0.20 +
            center          * 0.15
        )

        if lvs > best_score:
            best_score = lvs
            best_info = {
                "bbox": [float(v) for v in bbox],
                "size_ratio": float(size_ratio),
                "sharpness": float(sharpness),
                "frontality": float(front_score),
                "center": float(center),
                "num_players": len(detections),
            }

    return best_score, best_info


# ============================================================
# MAIN LOOP: Read video → sample → detect → score
# ============================================================
cap = cv2.VideoCapture(str(VIDEO_PATH))
frame_scores = []   # list of (frame_num, timestamp_sec, score, info)
frame_num = 0
batch_frames = []
batch_frame_nums = []

pbar = tqdm(total=TOTAL_FRAMES, desc="Scanning video", unit="frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    pbar.update(1)

    # Temporal sampling
    if frame_num % frame_interval != 0:
        frame_num += 1
        continue

    batch_frames.append(frame)
    batch_frame_nums.append(frame_num)

    # Process batch when full
    if len(batch_frames) >= YOLO_BATCH_SIZE:
        # Batch inference on GPU
        results = model.predict(
            batch_frames,
            classes=[0],  # person only
            conf=0.4,
            device=device,
            verbose=False,
        )

        for i, (fn, fr, res) in enumerate(zip(batch_frame_nums, batch_frames, results)):
            # Extract person detections
            dets = []
            if res.boxes is not None and len(res.boxes) > 0:
                for j in range(len(res.boxes)):
                    box = res.boxes.xyxy[j].cpu().numpy()
                    conf_val = float(res.boxes.conf[j].cpu().numpy())
                    dets.append([*box, conf_val])

            score, info = score_frame(fr, dets, WIDTH, HEIGHT)

            if score > 0:
                timestamp = fn / FPS
                frame_scores.append((fn, timestamp, score, info))

        batch_frames = []
        batch_frame_nums = []

    frame_num += 1

# Process remaining batch
if batch_frames:
    results = model.predict(
        batch_frames,
        classes=[0],
        conf=0.4,
        device=device,
        verbose=False,
    )
    for fn, fr, res in zip(batch_frame_nums, batch_frames, results):
        dets = []
        if res.boxes is not None and len(res.boxes) > 0:
            for j in range(len(res.boxes)):
                box = res.boxes.xyxy[j].cpu().numpy()
                conf_val = float(res.boxes.conf[j].cpu().numpy())
                dets.append([*box, conf_val])
        score, info = score_frame(fr, dets, WIDTH, HEIGHT)
        if score > 0:
            timestamp = fn / FPS
            frame_scores.append((fn, timestamp, score, info))

pbar.close()
cap.release()

print(f"\nScored {len(frame_scores):,} frames with players")
print(f"Score range: {min(s[2] for s in frame_scores):.3f} — {max(s[2] for s in frame_scores):.3f}")

## 4. Score Distribution & Top Frames Selection

In [ ]:
# Visualize score distribution
scores_only = [s[2] for s in frame_scores]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(scores_only, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=sorted(scores_only, reverse=True)[min(TARGET_FRAMES, len(scores_only))-1],
                color='red', linestyle='--', label=f'Top {TARGET_FRAMES} threshold')
axes[0].set_xlabel("Logo Visibility Score (LVS)")
axes[0].set_ylabel("Frame count")
axes[0].set_title("Score Distribution")
axes[0].legend()

# Score over time
timestamps = [s[1]/60 for s in frame_scores]  # in minutes
axes[1].scatter(timestamps, scores_only, s=1, alpha=0.3, c='steelblue')
axes[1].set_xlabel("Time (minutes)")
axes[1].set_ylabel("LVS")
axes[1].set_title("Score Over Match Timeline")

plt.tight_layout()
plt.show()

# Stats
print(f"Total scored frames: {len(frame_scores):,}")
print(f"Mean LVS: {np.mean(scores_only):.3f}")
print(f"Median LVS: {np.median(scores_only):.3f}")
print(f"Top 10% threshold: {np.percentile(scores_only, 90):.3f}")
print(f"Top 5% threshold: {np.percentile(scores_only, 95):.3f}")

## 5. De-duplicate Top Frames (Diversity Clustering)

From the top-scored frames, remove near-duplicates using perceptual hashing.
This ensures we get **diverse** scenes — not 50 frames from the same close-up replay.

In [ ]:
# ============================================================
# STEP 3: Sort by score → take top candidates → de-duplicate
# ============================================================
DEDUP_HASH_THRESHOLD = 6   # Min perceptual hash distance between selected frames
CANDIDATE_POOL = TARGET_FRAMES * 4  # Take 4x pool to allow de-dup filtering

# Sort by score descending
sorted_scores = sorted(frame_scores, key=lambda x: x[2], reverse=True)
top_candidates = sorted_scores[:CANDIDATE_POOL]

print(f"Top {CANDIDATE_POOL} candidates selected (score >= {top_candidates[-1][2]:.3f})")
print(f"De-duplicating with hash threshold = {DEDUP_HASH_THRESHOLD}...")

# Re-read video to extract only the top candidate frames
# This is much faster than reading all frames
candidate_frame_nums = set(c[0] for c in top_candidates)

cap = cv2.VideoCapture(str(VIDEO_PATH))
candidate_frames = {}  # frame_num → frame image

frame_num = 0
pbar = tqdm(total=TOTAL_FRAMES, desc="Reading top candidates", unit="frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    pbar.update(1)

    if frame_num in candidate_frame_nums:
        candidate_frames[frame_num] = frame

    # Stop early if we have all candidates
    if len(candidate_frames) == len(candidate_frame_nums):
        break

    frame_num += 1

pbar.close()
cap.release()

print(f"Loaded {len(candidate_frames)} candidate frames into memory")

# De-duplicate using perceptual hashing
selected = []
selected_hashes = []

for fn, ts, score, info in top_candidates:
    if fn not in candidate_frames:
        continue

    frame = candidate_frames[fn]
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)
    current_hash = imagehash.phash(pil_img)

    # Check if too similar to any already selected frame
    is_duplicate = False
    for existing_hash in selected_hashes:
        if current_hash - existing_hash < DEDUP_HASH_THRESHOLD:
            is_duplicate = True
            break

    if not is_duplicate:
        selected.append((fn, ts, score, info))
        selected_hashes.append(current_hash)

    if len(selected) >= TARGET_FRAMES:
        break

print(f"\nAfter de-duplication: {len(selected)} unique frames selected")
print(f"Score range: {selected[-1][2]:.3f} — {selected[0][2]:.3f}")

## 6. Save Best Frames with Timestamps

In [ ]:
# ============================================================
# STEP 4: Save selected frames + metadata CSV (all on Google Drive)
# ============================================================
def format_timestamp(sec):
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    return f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"

# Sort selected frames by timestamp (chronological order)
selected.sort(key=lambda x: x[0])

# Save frames and build metadata
metadata_rows = []

for idx, (fn, ts, score, info) in enumerate(tqdm(selected, desc="Saving best frames to Drive")):
    ts_str = format_timestamp(ts)
    filename = f"frame_{idx+1:04d}_{ts_str}_s{score:.2f}.jpg"

    frame = candidate_frames[fn]
    cv2.imwrite(str(BEST_FRAMES_DIR / filename), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

    metadata_rows.append({
        "frame_id": idx + 1,
        "filename": filename,
        "original_frame_num": fn,
        "timestamp_sec": round(ts, 2),
        "timestamp_hms": ts_str,
        "lvs_score": round(score, 4),
        "player_size_pct": round(info["size_ratio"] * 100, 2),
        "jersey_sharpness": round(info["sharpness"], 1),
        "frontality": round(info["frontality"], 2),
        "center_score": round(info["center"], 2),
        "num_players": info["num_players"],
        "source_video": VIDEO_PATH.name,
    })

# Save metadata CSV to Google Drive
df_meta = pd.DataFrame(metadata_rows)
csv_path = METADATA_DIR / "best_frames_index.csv"
df_meta.to_csv(csv_path, index=False)

print(f"\nSaved {len(selected)} frames to Google Drive:")
print(f"  Frames: {BEST_FRAMES_DIR}")
print(f"  CSV:    {csv_path}")
print(f"\nData persists across Colab sessions — no need to re-process!")

# Free memory
del candidate_frames
torch.cuda.empty_cache() if device == "cuda" else None

## 7. Preview Best Frames

Visual check: these should show **close-up, sharp, front-facing** players with clearly visible logos.

In [ ]:
# Preview: Top 12 highest-scored frames
best_files = sorted(BEST_FRAMES_DIR.glob("frame_*.jpg"))

# Sort by score (extracted from filename)
def extract_score(path):
    # filename format: frame_0001_02m15s_s0.85.jpg
    return float(path.stem.split("_s")[-1])

best_by_score = sorted(best_files, key=extract_score, reverse=True)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, fpath in enumerate(best_by_score[:12]):
    img = cv2.imread(str(fpath))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    score = extract_score(fpath)
    ts = fpath.stem.split("_")[2]  # timestamp part
    axes[i].set_title(f"LVS={score:.2f} | {ts}", fontsize=9)
    axes[i].axis("off")

plt.suptitle("Top 12 Frames by Logo Visibility Score", fontsize=14)
plt.tight_layout()
plt.show()

# Preview: Worst 4 of the selected (for calibration)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, fpath in enumerate(best_by_score[-4:]):
    img = cv2.imread(str(fpath))
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    score = extract_score(fpath)
    axes[i].set_title(f"LVS={score:.2f} (lowest selected)", fontsize=9)
    axes[i].axis("off")

plt.suptitle("Lowest 4 Selected Frames (quality floor)", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Auto-Annotate with Grounding DINO (Optional)

Run zero-shot logo detection on the selected frames.
If results are good → upload WITH annotations to Roboflow → only need to review & fix.

In [ ]:
# Install a compatible stack for GroundingDINO + Transformers
# (Fixes: missing config file + BertModel.get_head_mask incompatibility)
!pip install -q \
  "autodistill>=0.1.7" \
  "autodistill-grounding-dino>=0.1.3" \
  "supervision>=0.20.0" \
  "transformers==4.41.2"

# Ensure GroundingDINO config/checkpoint exist where autodistill expects them
import os
from pathlib import Path
import urllib.request

CACHE_DIR = Path("/root/.cache/autodistill/groundingdino")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CACHE_DIR / "GroundingDINO_SwinT_OGC.py"
CKPT_PATH = CACHE_DIR / "groundingdino_swint_ogc.pth"

CONFIG_URL = "https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py"
CKPT_URL = "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0/groundingdino_swint_ogc.pth"

if not CONFIG_PATH.exists():
    urllib.request.urlretrieve(CONFIG_URL, CONFIG_PATH)
    print(f"Downloaded config → {CONFIG_PATH}")
else:
    print(f"Config exists → {CONFIG_PATH}")

if not CKPT_PATH.exists():
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
    print(f"Downloaded checkpoint → {CKPT_PATH}")
else:
    print(f"Checkpoint exists → {CKPT_PATH}")

print("Done. If you still see import errors, Runtime → Restart session and run this cell again.")

In [ ]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

# Text prompts → label mapping
ontology = CaptionOntology({
    "AON logo on jersey": "aon",
    "AON text on shirt": "aon",
    "MCP logo": "mcp",
    "Cedar Court Hotels logo": "cch_cedar_court",
    "ChadLaw logo text": "chadlaw",
    "ATM Hospitality logo": "atm_hospitality",
    "EM workwear logo": "em_workwear",
    "Fairway Flooring logo": "fairway_flooring",
    "KLG logo text": "klg",
    "MNA logo text": "mna_cladding",
    "Bartercard logo": "bartercard",
    "Top Notch logo": "top_notch",
    "Romantica logo": "romantica_beds",
    "sponsor logo on rugby shirt": "sponsor_logo",
})

# Input: best_frames on Drive → Output: auto_annotated on Drive
base_model = GroundingDINO(
    ontology=ontology,
    box_threshold=0.20,
    text_threshold=0.15,
)

base_model.label(
    input_folder=str(BEST_FRAMES_DIR),
    output_folder=str(AUTOLABEL_DIR),
)

print("Auto-annotation complete!")
print(f"  Input:  {BEST_FRAMES_DIR}")
print(f"  Output: {AUTOLABEL_DIR}")
print("  All saved on Google Drive")

## 9. Upload to Roboflow (with pre-annotations)

In [ ]:
# ============================================================
# ROBOFLOW UPLOAD
# ============================================================
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your key
PROJECT_NAME = "kit-sponsor-logos"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()
project = workspace.project(PROJECT_NAME)

# Find annotated files on Google Drive
img_dir = AUTOLABEL_DIR / "train" / "images"
lbl_dir = AUTOLABEL_DIR / "train" / "labels"
if not img_dir.exists():
    img_dir = AUTOLABEL_DIR / "images"
    lbl_dir = AUTOLABEL_DIR / "labels"

annotated_images = sorted(img_dir.glob("*.jpg"))
print(f"Uploading {len(annotated_images)} frames from Google Drive to Roboflow...")

success, errors = 0, 0
for img_path in tqdm(annotated_images, desc="Uploading"):
    lbl_path = lbl_dir / (img_path.stem + ".txt")
    try:
        if lbl_path.exists() and lbl_path.read_text().strip():
            project.upload(image_path=str(img_path), annotation_path=str(lbl_path), split="train")
        else:
            project.upload(image_path=str(img_path), split="train")
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {e}")

print(f"\nDone! Success: {success} | Errors: {errors}")
print(f"Go to Roboflow to review: https://app.roboflow.com/{workspace.url}/{PROJECT_NAME}/annotate")

## 10. Summary

### What this notebook did:
```
162,000 frames (1.5h @ 30FPS)
    ↓ Temporal sampling (2 FPS)
~10,000 frames
    ↓ YOLOv8 person detection + Logo Visibility Score
~5,000 frames with players
    ↓ Sort by LVS score (top N)
~1,200 top candidates
    ↓ Perceptual hash de-duplication
~300 unique, high-quality frames
    ↓ Grounding DINO auto-annotation
~300 frames with pre-drawn bounding boxes
    ↓ Upload to Roboflow
Ready for human review & correction
```

### Reduction: 162,000 → 300 frames (99.8% reduction)

### Google Drive structure:
```
Google Drive/Bradford_Bulls/
├── videos/              ← Downloaded match videos (reusable)
├── best_frames/         ← Top 300 selected frames
├── metadata/            ← best_frames_index.csv (scores, timestamps)
├── auto_annotated/      ← Grounding DINO labels (YOLO format)
└── dataset/             ← Annotated dataset from Roboflow (Phase 2)
```

All data persists on Google Drive — if Colab disconnects, just remount and continue.

### Next steps:
1. Review annotations on Roboflow (fix wrong labels, add missed logos)
2. Generate dataset version on Roboflow
3. Run `colab_02_train_model.ipynb` to fine-tune YOLOv8